<a href="https://colab.research.google.com/github/Innovatewithapple/SKLearnRegressionProblems/blob/main/CarPrice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import os
from google.colab import files,userdata
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_validate,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error,r2_score,mean_squared_error,mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor

In [2]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('key activated')

key activated


In [3]:
!kaggle datasets download -d hellbuoy/car-price-prediction

Dataset URL: https://www.kaggle.com/datasets/hellbuoy/car-price-prediction
License(s): unknown
100% 18.1k/18.1k [00:00<00:00, 28.4MB/s]



In [4]:
!unzip -q /content/car-price-prediction.zip -d /content/

In [6]:
df = pd.read_csv('/content/CarPrice_Assignment.csv')
df.sample(5)

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
46,47,2,isuzu D-Max,gas,std,two,hatchback,rwd,front,96.0,...,119,spfi,3.43,3.23,9.2,90,5000,24,29,11048.0
74,75,1,buick regal sport coupe (turbo),gas,std,two,hardtop,rwd,front,112.0,...,304,mpfi,3.80,3.35,8.0,184,4500,14,16,45400.0
182,183,2,vokswagen rabbit,diesel,std,two,sedan,fwd,front,97.3,...,97,idi,3.01,3.40,23.0,52,4800,37,46,7775.0
61,62,1,mazda glc custom,gas,std,two,hatchback,fwd,front,98.8,...,122,2bbl,3.39,3.39,8.6,84,4800,26,32,10595.0
97,98,1,nissan note,gas,std,four,wagon,fwd,front,94.5,...,97,2bbl,3.15,3.29,9.4,69,5200,31,37,7999.0


In [7]:
df.isnull().sum()

,0
car_ID,0
symboling,0
CarName,0
fueltype,0
aspiration,0
doornumber,0
carbody,0
drivewheel,0
enginelocation,0
wheelbase,0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   CarName           205 non-null    object 
 3   fueltype          205 non-null    object 
 4   aspiration        205 non-null    object 
 5   doornumber        205 non-null    object 
 6   carbody           205 non-null    object 
 7   drivewheel        205 non-null    object 
 8   enginelocation    205 non-null    object 
 9   wheelbase         205 non-null    float64
 10  carlength         205 non-null    float64
 11  carwidth          205 non-null    float64
 12  carheight         205 non-null    float64
 13  curbweight        205 non-null    int64  
 14  enginetype        205 non-null    object 
 15  cylindernumber    205 non-null    object 
 16  enginesize        205 non-null    int64  
 1

In [9]:
df.describe()

,car_ID,symboling,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
count,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000,205.000000
mean,103.000000,0.834146,98.756585,174.049268,65.907805,53.724878,2555.565854,126.907317,3.329756,3.255415,10.142537,104.117073,5125.121951,25.219512,30.751220,13276.710571
std,59.322565,1.245307,6.021776,12.337289,2.145204,2.443522,520.680204,41.642693,0.270844,0.313597,3.972040,39.544167,476.985643,6.542142,6.886443,7988.852332
min,1.000000,-2.000000,86.600000,141.100000,60.300000,47.800000,1488.000000,61.000000,2.540000,2.070000,7.000000,48.000000,4150.000000,13.000000,16.000000,5118.000000
25%,52.000000,0.000000,94.500000,166.300000,64.100000,52.000000,2145.000000,97.000000,3.150000,3.110000,8.600000,70.000000,4800.000000,19.000000,25.000000,7788.000000
50%,103.000000,1.000000,97.000000,173.200000,65.500000,54.100000,2414.000000,120.000000,3.310000,3.290000,9.000000,95.000000,5200.000000,24.000000,30.000000,10295.000000
75%,154.000000,2.000000,102.400000,183.100000,66.900000,55.500000,2935.000000,141.000000,3.580000,3.410000,9.400000,116.000000,5500.000000,30.000000,34.000000,16503.000000
max,205.000000,3.000000,120.900000,208.100000,72.300000,59.800000,4066.000000,326.000000,3.940000,4.170000,23.000000,288.000000,6600.000000,49.000000,54.000000,45400.000000


In [33]:
x = df.drop(columns=['price','car_ID','CarName'])
y = df['price']

In [34]:
num_cols = x.select_dtypes(include=['int64','float64']).columns
cat_cols = x.select_dtypes(include='object').columns

In [35]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [36]:
preprocessor = ColumnTransformer([
    ('nums',StandardScaler(),num_cols),
    ('cats',OneHotEncoder(handle_unknown='ignore'),cat_cols)
])

In [37]:
#linear Regression Pipeline
linearPipeLine = Pipeline([
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
])

In [38]:
linearPipeLine.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  Index(['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight',
       'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio',
       'horsepower', 'peakrpm', 'citympg', 'highwaympg'],
      dtype='object')),
                                                 ('cats',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel',
       'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem'],
      dtype='object'))])),
                ('model', LinearRegression())])

In [39]:
linear_cross_validate = cross_validate(linearPipeLine,x_train,y_train,cv=5,scoring='r2',error_score='raise')
linear_cross_validate

{'fit_time': array([0.01686525, 0.01096654, 0.01032615, 0.01054502, 0.01102304]),
 'score_time': array([0.00684071, 0.0065856 , 0.00700903, 0.00804615, 0.00631475]),
 'test_score': array([0.69836847, 0.7974113 , 0.93191793, 0.80503647, 0.87965225])}

In [41]:
print('R2 Mean: ',linear_cross_validate['test_score'].mean())
print('R2 std: ',linear_cross_validate['test_score'].std())

R2 Mean:  0.8224772862000227
R2 std:  0.07947542454924245


In [42]:
#SVR
svr_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',SVR(kernel='rbf'))
])

In [44]:
svr_pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  Index(['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight',
       'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio',
       'horsepower', 'peakrpm', 'citympg', 'highwaympg'],
      dtype='object')),
                                                 ('cats',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel',
       'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem'],
      dtype='object'))])),
                ('model', SVR())])

In [45]:
svr_cross_score = cross_validate(svr_pipeline,x_train,y_train,cv=5,scoring='r2')
svr_cross_score

{'fit_time': array([0.04220033, 0.07233262, 0.03932047, 0.056041  , 0.08760047]),
 'score_time': array([0.046803  , 0.02321863, 0.02304339, 0.05164242, 0.03644824]),
 'test_score': array([-0.05671549, -0.0431942 , -0.04628269, -0.31217788, -0.16828854])}

In [46]:
param_grid = {
    # We move from tiny C to massive C
    'model__C': [1, 100, 1000, 10000, 50000],
    'model__epsilon': [0.1, 1, 10, 100, 1000],
    'model__kernel': ['rbf']
}

grid_svr = GridSearchCV(svr_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_svr.fit(x_train, y_train)

print(f"Best Params: {grid_svr.best_params_}")
print(f"Best Score: {grid_svr.best_score_}")

Best Params: {'model__C': 50000, 'model__epsilon': 1000, 'model__kernel': 'rbf'}
Best Score: 0.858099072055379


In [47]:
randomForest_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestRegressor())
])

In [48]:
randomForest_cross_score = cross_validate(randomForest_pipeline,x_train,y_train,cv=5,scoring='r2')
print('R2 Mean: ',randomForest_cross_score['test_score'].mean())
print('R2 STD: ',randomForest_cross_score['test_score'].std())

R2 Mean:  0.894104254477066
R2 STD:  0.03655555884067154


In [49]:
rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 15, None],
    'model__min_samples_split': [2, 5, 10], # <--- Control the "starting" of a split
    'model__min_samples_leaf': [1, 2, 4],   # <--- Control the "size" of the final group
    'model__max_features': ['sqrt', None]
}

In [50]:
randomforest_Grid = GridSearchCV(randomForest_pipeline,rf_param_grid,cv=2,scoring='r2')
randomforest_Grid.fit(x_train,y_train)

GridSearchCV(cv=2,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('nums',
                                                                         StandardScaler(),
                                                                         Index(['symboling', 'wheelbase', 'carlength', 'carwidth', 'carheight',
       'curbweight', 'enginesize', 'boreratio', 'stroke', 'compressionratio',
       'horsepower', 'peakrpm', 'citympg', 'highwaympg'],
      dtype='object')),
                                                                        ('cats',
                                                                         OneHotEncoder(handle_unknown=...
                                                                         Index(['fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel',
       'enginelocation', 'enginetype', 'cylindernumber', 'fuelsystem'],
      dtype='object'))])),
                                       ('model', RandomForestRegressor())]),
             param_grid={'model__max_depth': [10, 15, None],
                         'model__max_features': ['sqrt', None],
                         'model__min_samples_leaf': [1, 2, 4],
                         'model__min_samples_split': [2, 5, 10],
                         'model__n_estimators': [100, 200]},
             scoring='r2')

In [51]:
print(randomforest_Grid.best_params_)

{'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}


In [52]:
y_pred_forest = randomforest_Grid.best_estimator_.predict(x_test)

# 2. Check the Final Scores
print(f"Final R2 Score: {r2_score(y_test, y_pred_forest)}")
print(f"Final MAE: ${mean_absolute_error(y_test, y_pred_forest):.2f}")

Final R2 Score: 0.9436452766249768
Final MAE: $1411.91


In [55]:
mape = mean_absolute_percentage_error(y_test, y_pred_forest)
print("MAPE:", mape * 100, "%")

MAPE: 11.031146131771207 %


In [56]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred_forest))
print("RMSE:", rmse)

RMSE: 1975.9812728993436
